# Data Agent Platform Studio — Colab GPU / CPU

A self-contained Data Agent Studio using one shared Bronze/Silver/Gold pipeline and two CrewAI agents. Both agents share one local Hugging Face model. CUDA (T4 target) is recommended; CPU-only inference is also supported (slower, FP32).

No external hosted-agent or vendor-agent API is used. Select a GPU runtime when available, or use the standard CPU runtime. Set `HF_DEVICE=cpu` to force CPU or `HF_DEVICE=auto` to prefer CUDA and fall back to CPU.

In [ ]:
%cd /content
!if [ -d data-agent-platform-studio/.git ]; then cd data-agent-platform-studio && git fetch origin feature/colab-crewai-dual-agent-studio && git checkout feature/colab-crewai-dual-agent-studio && git pull --ff-only origin feature/colab-crewai-dual-agent-studio; else git clone -b feature/colab-crewai-dual-agent-studio https://github.com/msellamiTN/data-agent-platform-studio.git; fi
%cd /content/data-agent-platform-studio
!git rev-parse --short HEAD

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
VENV_DIR = PROJECT_ROOT / ".venv"
VENV_PYTHON = VENV_DIR / "bin" / "python"

# Keep Colab's preinstalled CUDA-enabled PyTorch, but isolate all Studio packages.
venv_config = VENV_DIR / "pyvenv.cfg"
needs_venv = (
    not VENV_PYTHON.is_file()
    or not venv_config.is_file()
    or "include-system-site-packages = true" not in venv_config.read_text().lower()
)
if needs_venv:
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)
    subprocess.check_call([
        sys.executable, "-m", "venv",
        "--system-site-packages", str(VENV_DIR),
    ])

# Install uv without asking pip to resolve/modify Colab's unrelated base packages.
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--quiet", "--no-deps", "uv",
])
uv_executable = shutil.which("uv") or str(Path(sys.prefix) / "bin" / "uv")
if not Path(uv_executable).is_file() and shutil.which(uv_executable) is None:
    raise RuntimeError("uv installer completed but the uv executable was not found.")

# Resolve only the Studio's dependency graph inside .venv. No Colab base packages
# such as Google ADK, BigFrames, GCSFS, Datasets, Numba or Tweepy are downgraded.
subprocess.check_call([
    uv_executable, "pip", "install",
    "--python", str(VENV_PYTHON),
    "-r", str(PROJECT_ROOT / "requirements-colab.txt"),
])

runtime_probe = r'''
import json
import torch
import transformers
import huggingface_hub
import gradio
import crewai
import accelerate
import pandas
import plotly
from transformers import AutoModelForCausalLM, AutoTokenizer
from packaging.version import Version

if transformers.__version__ != "5.24.1":
    raise RuntimeError(f"Expected transformers 5.24.1, found {transformers.__version__}")
if huggingface_hub.__version__ != "1.32.0":
    raise RuntimeError(f"Expected huggingface-hub 1.32.0, found {huggingface_hub.__version__}")
if Version(torch.__version__.split("+")[0]) < Version("2.6.0"):
    raise RuntimeError(f"PyTorch >= 2.6 is required, found {torch.__version__}")

print(json.dumps({
    "python": __import__("platform").python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "gradio": gradio.__version__,
    "crewai": getattr(crewai, "__version__", "installed"),
    "accelerate": accelerate.__version__,
    "pandas": pandas.__version__,
    "plotly": plotly.__version__,
    "model_api_imports": "passed",
    "torch_source": torch.__file__,
}, indent=2))
'''
subprocess.check_call([str(VENV_PYTHON), "-c", runtime_probe])

# Pass an explicit device choice to the isolated launch process.
os.environ.setdefault("HF_DEVICE", "auto" if __import__("torch").cuda.is_available() else "cpu")
os.environ.setdefault("HF_CPU_THREADS", "4")
print("Studio environment ready:", VENV_PYTHON)


In [ ]:
import os, platform, torch
from packaging.version import Version

print("Notebook Python:", platform.python_version())
print("Colab base PyTorch:", torch.__version__)
if Version(torch.__version__.split("+")[0]) < Version("2.6.0"):
    raise RuntimeError(
        "PyTorch >= 2.6 is required. Select a current Colab runtime; "
        "the notebook will not replace Colab's CUDA-specific PyTorch."
    )
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    os.environ.setdefault("HF_DEVICE", "auto")
else:
    os.environ["HF_DEVICE"] = "cpu"
    print("Using CPU inference (FP32). Expect slower generation and higher RAM use.")
    print("CPU threads:", os.getenv("HF_CPU_THREADS", "4"))


In [ ]:
import subprocess
from pathlib import Path

venv_python = Path(".venv/bin/python").resolve()
subprocess.check_call([
    str(venv_python), "-c",
    "from studio.config import MODEL_ID; import os; "
    "print('Hugging Face model:', MODEL_ID); "
    "print('Selected device:', os.getenv('HF_DEVICE', 'auto')); "
    "print('Precision: FP16 on CUDA; FP32 on CPU')",
])


## Live runtime acceptance and launch

The isolated Studio process loads the local model, runs one inference, and requires one real CrewAI invocation of a registered Python tool. If both checks pass, it launches Gradio in the same process using the already-loaded model. This avoids a second model download/load and keeps Colab's unrelated preinstalled packages untouched.

The launch URL is public to anyone who obtains it. Use synthetic data only.

In [ ]:
!./.venv/bin/python scripts/launch_colab.py